## Extract: Process to pull data from Source system
## Load: Process to write data to a destination system

* Common upstream & downstream systems
1. OLTP(Online Transaction Processing) Databases: Postgres, MySQL, sqlite3, etc
2. OLAP(Online Analytical Processing) Databases: Snowflake, BigQuery, Clickhouse, DuckDB, etc
3. Cloud data storage: AWS S3, GCP Cloud Store, Minio, etc
4. Queue systems: Kafka, Redpanda, etc
5. API
6. Local disk: csv, excel, json, xml files
7. SFTP\FTP server

In [ ]:
# Databases: When reading or writing to a database we use a database driver. Database drivers are libraries that we can use to read or write to a database.
# Question: How do you read data from a sqlite3 database and write to a DuckDB database?
# Hint: Look at importing the database libraries for sqlite3 and duckdb and create connections to talk to the respective databases
import sqlite3
import duckdb

sqlite3_conn = sqlite3.connect('tpch.db')
duckdb_conn = duckdb.connect('duckdb.db')

sql_cursor = sqlite3_conn.cursor()
duck_cursor = duckdb_conn.cursor()

# Fetch data from the SQLite Customer table
customer_data = sql_cursor.execute("SELECT * FROM Customer").fetchall()

# Insert data into the DuckDB Customer table
insert_query = f'''INSERT INTO Customer (customer_id, zipcode, city, state_code, datetime_created, datetime_updated)
VALUES (?,?,?,?,?,?)'''

if len(customer_data)>0:
    duck_cursor.executemany(insert_query,customer_data)

    # Hint: Look for Commit and close the connections
    # Commit tells the DB connection to send the data to the database and commit it, if you don't commit the data will not be inserted
    duck_cursor.commit()

# We should close the connection, as DB connections are expensive
sqlite3_conn.close()
duckdb_conn.close()


# Cloud storage

In [ ]:

# Question: How do you read data from the S3 location given below and write the data to a DuckDB database?
# Data source: https://docs.opendata.aws/noaa-ghcn-pds/readme.html station data at path "csv.gz/by_station/ASN00002022.csv.gz"
# Hint: Use boto3 client with UNSIGNED config to access the S3 bucket
# Hint: The data will be zipped you have to unzip it and decode it to utf-8
import csv
import gzip
from io import StringIO

import boto3 # Mainly used to connect with the AWS S3
import duckdb
from botocore import UNSIGNED # To access AWS data without credentials
from botocore.client import Config # Finetuning the AWS client behaviour

# AWS S3 bucket and file details
bucket_name = "noaa-ghcn-pds"
file_key = "csv.gz/by_station/ASN00002022.csv.gz"
# Create a boto3 client with anonymous access
s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# Download the CSV file from S3
response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
compressed_data = response["Body"].read()

# Decompress the gzip data
csv_data = gzip.decompress(compressed_data).decode("utf-8")

# Read the CSV file using csv.reader
csv_reader = csv.reader(StringIO(csv_data))
data = list(csv_reader)

# Connect to the DuckDB database (assume WeatherData table exists)
duckdb_conn = duckdb.connect("duckdb.db")

# Insert data into the DuckDB WeatherData table
insert_query = '''INSERT INTO WeatherData (id, date, element, value, m_flag, q_flag, s_flag, obs_time)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
'''
duckdb_conn.executemany(insert_query, data[:1000])

duckdb_conn.commit()
duckdb_conn.close()

# API

In [ ]:

# Question: How do you read data from the CoinCap API given below and write the data to a DuckDB database?
# URL: "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd"
# Hint: use requests library

# Define the API endpoint
url = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd"

# Fetch data from the CoinCap API
import requests
import duckdb
import json

response = requests.get(url)
data = response.json()[1]
print(data)
# Data dict to list
data_l = []
for k,v in data.items():
    if k == 'roi':
        data_l.append(json.dumps(v))
    else:
        data_l.append(v)
print(data_l)

# Connect to the DuckDB database
duckdb_conn = duckdb.connect("duckdb.db")
# Insert data into the DuckDB Exchanges table
insert_query = '''
INSERT INTO Exchanges (  id,
    symbol,
    name,
    image,
    current_price,
    market_cap,
    market_cap_rank,
    fully_diluted_valuation,
    total_volume,
    high_24h,
    low_24h,
    price_change_24h,
    price_change_percentage_24h,
    market_cap_change_24h,
    market_cap_change_percentage_24h,
    circulating_supply,
    total_supply,
    max_supply,
    ath,
    ath_change_percentage,
    ath_date,
    atl,
    atl_change_percentage,
    atl_date,
    roi,
    last_updated)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
'''
# Prepare data for insertion
duck_cursor = duckdb_conn.cursor()
duck_cursor.execute(insert_query, data_l)
# Hint: Ensure that the data types of the data to be inserted is compatible with DuckDBs data column types in ./setup_db.py
duckdb_conn.commit()
duckdb_conn.close()


# Local disk

In [33]:

# Question: How do you read a CSV file from local disk and write it to a database?
# Look up open function with csvreader for python
import csv

duckdb_conn = duckdb.connect("duckdb.db")
duck_cursor = duckdb_conn.cursor()
insert_query = '''INSERT INTO Customer (customer_id, zipcode, city, state_code, datetime_created, datetime_updated)
                VALUES (?, ?, ?, ?, ?, ?)'''
with open('./data/customers.csv', "r") as file:
    reader = csv.DictReader(file) ## We have changed to DictReader from reader as it gives all in [String]
    rows = [
        (
        row['customer_id'],
        row['zipcode'],
        row['city'],
        row['state_code'],
        row['datetime_created'],
        row['datetime_updated']
    )
    for row in reader
    ]
    duck_cursor.executemany(insert_query, rows)
    duckdb_conn.commit()
    duckdb_conn.close()

# Web scraping

In [36]:
# Questions: Use beatiful soup to scrape the below website and print all the links in that website
# URL of the website to scrape
url = 'https://example.com'

import requests
from bs4 import BeautifulSoup
# GET the data from the wesite
response = requests.get(url)

# Parse HTML content of the webpage
soap = BeautifulSoup(response.text, 'html.parser')
print(soap)
# Example: Find and print all the links on the webpage
for links in soap.find_all('a'):
    print(links)
    print(links.get('href'))


<!DOCTYPE html>
<html lang="en"><head><title>Example Domain</title><meta content="width=device-width, initial-scale=1" name="viewport"/><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>

<a href="https://iana.org/domains/example">Learn more</a>
https://iana.org/domains/example
